# 07 - Multi-part removal pilot

This notebook runs a **preliminary multi-part removal pilot** for minimal-change local mesh repair.

Supported pilot settings:
- `chair_two_leg.yaml`
- `table_two_leg.yaml`

Compared methods in the exported summary:
- Planar + RPA
- Advancing-front + RPA
- Planar + LH-only

Main reported metrics:
- Residual
- Improvement
- Locality
- Avg Quality
- CD
- HD

This notebook **does not rewrite the experiment pipeline**. It directly calls the existing Python code in `src/experiments/run_batch.py`, then aggregates the wide-format result table into a compact paper-ready summary.


In [49]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Robust project-root detection: walk upward until both src/ and configs/ are found
CUR = Path.cwd().resolve()
PROJECT_ROOT = CUR

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "configs").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Cannot find project root from {CUR}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD         =", CUR)
print("PROJECT_ROOT=", PROJECT_ROOT)
print("Has src     =", (PROJECT_ROOT / "src").exists())
print("Has configs =", (PROJECT_ROOT / "configs").exists())

CWD         = D:\MyJupyter\Works\3DPART_v2\notebooks
PROJECT_ROOT= D:\MyJupyter\Works\3DPART_v2
Has src     = True
Has configs = True


In [50]:
# Select pilot config here
#CONFIG_NAME = "chair_two_leg.yaml"
CONFIG_NAME = "table_two_leg.yaml"

CONFIG_PATH = PROJECT_ROOT / "configs" / CONFIG_NAME
print("Using config:", CONFIG_PATH)

Using config: D:\MyJupyter\Works\3DPART_v2\configs\table_two_leg.yaml


In [51]:
from src.config import load_config, ensure_dirs
from src.data.dataset_index import DatasetIndex
from src.experiments.run_batch import run_batch_experiment

cfg = load_config(str(CONFIG_PATH))
ensure_dirs(cfg)

EXP_NAME = cfg["experiment"]["name"]
RAW_DATA_DIR = Path(cfg["paths"]["raw_data_dir"])
OUTPUTS_ROOT = PROJECT_ROOT / "outputs"
PILOT_OUTPUT_DIR = OUTPUTS_ROOT / EXP_NAME
PILOT_METRICS_DIR = PILOT_OUTPUT_DIR / "metrics"
TABLES_DIR = OUTPUTS_ROOT / "tables"

PILOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PILOT_METRICS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment name :", EXP_NAME)
print("Category        :", cfg["dataset"]["category"])
print("Semantic label  :", cfg["dataset"]["semantic_label"])
print("Raw data dir    :", RAW_DATA_DIR)
print("Pilot output dir:", PILOT_OUTPUT_DIR)
print("num_parts_to_remove =", cfg["dataset"].get("num_parts_to_remove", 1))
print("part_selection_mode =", cfg["dataset"].get("part_selection_mode", "random_distinct"))

Experiment name : table_two_leg_removal
Category        : Table
Semantic label  : leg
Raw data dir    : D:\MyJupyter\Works\3DPART_v2\data\raw\table_two_leg_removal
Pilot output dir: D:\MyJupyter\Works\3DPART_v2\outputs\table_two_leg_removal
num_parts_to_remove = 2
part_selection_mode = random_distinct


In [52]:
# Quick dataset sanity check
if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data dir does not exist: {RAW_DATA_DIR}")

sample_dirs = [p for p in RAW_DATA_DIR.iterdir() if p.is_dir()]
print("Built sample dirs:", len(sample_dirs))
print("First 5 sample dirs:", [p.name for p in sample_dirs[:5]])

index = DatasetIndex(str(RAW_DATA_DIR))
sample_ids = index.sample_ids
print("Indexed sample count:", len(sample_ids))
print("First 10 sample IDs:", sample_ids[:10])

Built sample dirs: 20
First 5 sample dirs: ['18942', '19117', '20667', '21184', '21620']
Indexed sample count: 20
First 10 sample IDs: ['24270', '32881', '29101', '26430', '30908', '25724', '23706', '19117', '25445', '33988']


In [53]:
# Metadata sanity check: confirm this really is a multi-part dataset
meta_files = sorted(RAW_DATA_DIR.glob("*/meta.json"))
print("meta files:", len(meta_files))

preview_rows = []
removed_counts = []
for mf in meta_files[:10]:
    with open(mf, "r", encoding="utf-8") as f:
        meta = json.load(f)
    removed_counts.append(meta.get("removed_part_count"))
    preview_rows.append({
        "sample_id": meta.get("sample_id"),
        "removed_part_name": meta.get("removed_part_name"),
        "removed_part_count": meta.get("removed_part_count"),
        "n_removed_obj_files": len(meta.get("removed_obj_files", [])),
    })

display(pd.DataFrame(preview_rows))
print("\nremoved_part_count distribution (preview only):")
print(pd.Series(removed_counts).value_counts(dropna=False).sort_index())

meta files: 20


,sample_id,removed_part_name,removed_part_count,n_removed_obj_files
0,18942,leg,1,45
1,19117,leg,1,5
2,20667,leg,1,100
3,21184,leg,1,40
4,21620,leg,1,77
5,23706,leg,1,44
6,24270,leg,1,5
7,24325,leg,1,40
8,25445,leg,1,200
9,25724,leg,1,80



removed_part_count distribution (preview only):
1    10
Name: count, dtype: int64


## Run batch experiment

This calls the existing `run_batch_experiment(...)` function and writes results to:

- `outputs/<experiment_name>/metrics/all_results.csv`
- `outputs/<experiment_name>/metrics/run_summary.json`


In [54]:
# Optional: subsample for a quicker pilot run
# Set MAX_SAMPLES = None to use all built samples
MAX_SAMPLES = None
# MAX_SAMPLES = 20

if MAX_SAMPLES is not None:
    sample_ids_to_run = sample_ids[:MAX_SAMPLES]
else:
    sample_ids_to_run = sample_ids

print("Running on", len(sample_ids_to_run), "samples")

Running on 20 samples


In [71]:
df_results = run_batch_experiment(
    data_dir=str(RAW_DATA_DIR),
    output_dir=str(PILOT_OUTPUT_DIR),
    sample_ids=sample_ids_to_run,
    margin=cfg["repair"]["margin"],
    proximity_threshold=cfg["repair"]["proximity_threshold"],
    save_meshes=False,
    include_distance=True,
    include_sota=True,   # pilot uses local methods only
    mlp_generator=None,
)

print("df_results shape:", df_results.shape)
display(df_results.head())

2026-04-16 19:00:03 [INFO] BatchExperiment: Running on 20 samples (SOTA=True, distance=True, mlp=no)
Running experiments:   5%|██▉                                                        | 1/20 [04:10<1:19:28, 250.95s/it]2026-04-16 19:04:14 [WARNING] BatchExperiment: Sample 32881: No boundary loops found
2026-04-16 19:04:15 [WARNING] BatchExperiment: Sample 29101: No boundary loops found
Running experiments: 100%|████████████████████████████████████████████████████████████| 20/20 [41:25<00:00, 124.28s/it]
2026-04-16 19:41:29 [INFO] BatchExperiment: Done: 11 success, 9 failed


df_results shape: (11, 115)


,sample_id,n_boundary_loops,n_target_loops_rpa,center_fan/closure_residual,center_fan/improvement,center_fan/n_new_vertices,center_fan/n_new_faces,center_fan/avg_new_face_quality,center_fan/min_new_face_quality,center_fan/std_new_face_quality,...,advancing_front_rpa/std_new_face_quality,advancing_front_rpa/n_faces_inside,advancing_front_rpa/n_faces_outside,advancing_front_rpa/locality_ratio,advancing_front_rpa/chamfer_distance,advancing_front_rpa/hausdorff_distance,advancing_front_rpa/dev_mean,advancing_front_rpa/dev_max,advancing_front_rpa/dev_median,advancing_front_rpa/dev_std
0,24270,85,15,0.0,36.157610,15.0,2575.0,0.090408,0.003998,0.099210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,26430,60,60,0.0,11.481977,60.0,670.0,0.761807,0.291099,0.149943,...,0.163744,550.0,0.0,1.000000,0.044757,0.389519,0.008897,0.029275,0.008384,0.004578
2,30908,20,20,0.0,41.163587,20.0,2245.0,0.094986,0.010715,0.109541,...,0.115033,2205.0,0.0,1.000000,0.032497,0.176350,0.012867,0.062111,0.010638,0.009592
3,25724,25,25,0.0,98.979169,25.0,5670.0,0.043026,0.000462,0.101663,...,0.045451,120.0,5500.0,0.021352,0.105863,0.598820,0.011634,0.038924,0.010856,0.006058
4,23706,34,34,0.0,69.755712,34.0,3539.0,0.068751,0.005620,0.119755,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [72]:
RESULT_CSV = PILOT_METRICS_DIR / "all_results.csv"
SUMMARY_JSON = PILOT_METRICS_DIR / "run_summary.json"

print("RESULT_CSV =", RESULT_CSV)
print("SUMMARY_JSON =", SUMMARY_JSON)

if not RESULT_CSV.exists():
    raise FileNotFoundError(f"Expected result csv not found: {RESULT_CSV}")

df = pd.read_csv(RESULT_CSV)
display(df.head())

with open(SUMMARY_JSON, "r", encoding="utf-8") as f:
    run_summary = json.load(f)

print("Run summary:")
print(json.dumps(run_summary, indent=2, ensure_ascii=False))

RESULT_CSV = D:\MyJupyter\Works\3DPART_v2\outputs\table_two_leg_removal\metrics\all_results.csv
SUMMARY_JSON = D:\MyJupyter\Works\3DPART_v2\outputs\table_two_leg_removal\metrics\run_summary.json


,sample_id,n_boundary_loops,n_target_loops_rpa,center_fan/closure_residual,center_fan/improvement,center_fan/n_new_vertices,center_fan/n_new_faces,center_fan/avg_new_face_quality,center_fan/min_new_face_quality,center_fan/std_new_face_quality,...,advancing_front_rpa/std_new_face_quality,advancing_front_rpa/n_faces_inside,advancing_front_rpa/n_faces_outside,advancing_front_rpa/locality_ratio,advancing_front_rpa/chamfer_distance,advancing_front_rpa/hausdorff_distance,advancing_front_rpa/dev_mean,advancing_front_rpa/dev_max,advancing_front_rpa/dev_median,advancing_front_rpa/dev_std
0,24270,85,15,0.0,36.157610,15.0,2575.0,0.090408,0.003998,0.099210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,26430,60,60,0.0,11.481977,60.0,670.0,0.761807,0.291099,0.149943,...,0.163744,550.0,0.0,1.000000,0.044757,0.389519,0.008897,0.029275,0.008384,0.004578
2,30908,20,20,0.0,41.163587,20.0,2245.0,0.094986,0.010715,0.109541,...,0.115033,2205.0,0.0,1.000000,0.032497,0.176350,0.012867,0.062111,0.010638,0.009592
3,25724,25,25,0.0,98.979169,25.0,5670.0,0.043026,0.000462,0.101663,...,0.045451,120.0,5500.0,0.021352,0.105863,0.598820,0.011634,0.038924,0.010856,0.006058
4,23706,34,34,0.0,69.755712,34.0,3539.0,0.068751,0.005620,0.119755,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Run summary:
{
  "failed": [
    "32881",
    "29101",
    "25445",
    "21620",
    "26932",
    "31830",
    "28033",
    "20667",
    "18942"
  ],
  "n_success": 11,
  "n_failed": 9
}


## Aggregate wide-format results into a compact paper table

In [73]:
METHODS = {
    "planar_removed_part_aware": "Planar + RPA",
    "advancing_front_rpa": "Advancing-front + RPA",
    "planar_largest_hole_only": "Planar + LH-only",
}

METRICS = {
    "target_loop_length_after": "Residual ↓",
    "improvement": "Improvement ↑",
    "avg_new_face_quality": "Quality ↑",
    "locality_ratio": "Locality ↑",
    "chamfer_distance": "CD ↓",
    "hausdorff_distance": "HD ↓",
}

def summarize_methods(df_wide, methods=METHODS, metrics=METRICS):
    rows = []
    for method_key, display_name in methods.items():
        row = {"Method": display_name}
        for metric_key, display_metric in metrics.items():
            col = f"{method_key}/{metric_key}"
            if col in df_wide.columns:
                row[display_metric] = pd.to_numeric(df_wide[col], errors="coerce").mean()
            else:
                row[display_metric] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

pilot_table = summarize_methods(df)
pilot_table

,Method,Residual ↓,Improvement ↑,Quality ↑,Locality ↑,CD ↓,HD ↓
0,Planar + RPA,NaN,51.213639,0.461761,0.517594,0.047981,0.353120
1,Advancing-front + RPA,NaN,57.516401,0.491303,0.510707,0.050260,0.367062
2,Planar + LH-only,NaN,4.127725,0.472098,0.430828,0.046806,0.358677


In [74]:
# Round for readability
pilot_table_rounded = pilot_table.copy()
for c in pilot_table_rounded.columns:
    if c != "Method":
        pilot_table_rounded[c] = pilot_table_rounded[c].map(lambda x: round(float(x), 4) if pd.notnull(x) else x)

pilot_table_rounded

,Method,Residual ↓,Improvement ↑,Quality ↑,Locality ↑,CD ↓,HD ↓
0,Planar + RPA,NaN,51.2136,0.4618,0.5176,0.0480,0.3531
1,Advancing-front + RPA,NaN,57.5164,0.4913,0.5107,0.0503,0.3671
2,Planar + LH-only,NaN,4.1277,0.4721,0.4308,0.0468,0.3587


In [75]:
# Save summary tables
out_csv = TABLES_DIR / f"{EXP_NAME}_multipart_pilot_summary.csv"
out_tex = TABLES_DIR / f"{EXP_NAME}_multipart_pilot_summary.tex"

pilot_table_rounded.to_csv(out_csv, index=False)

latex_str = pilot_table_rounded.to_latex(index=False, float_format=lambda x: f"{x:.4f}")
with open(out_tex, "w", encoding="utf-8") as f:
    f.write(latex_str)

print("Saved CSV :", out_csv)
print("Saved TeX :", out_tex)
print("\nLaTeX preview:\n")
print(latex_str)

Saved CSV : D:\MyJupyter\Works\3DPART_v2\outputs\tables\table_two_leg_removal_multipart_pilot_summary.csv
Saved TeX : D:\MyJupyter\Works\3DPART_v2\outputs\tables\table_two_leg_removal_multipart_pilot_summary.tex

LaTeX preview:

\begin{tabular}{lrrrrrr}
\toprule
Method & Residual ↓ & Improvement ↑ & Quality ↑ & Locality ↑ & CD ↓ & HD ↓ \\
\midrule
Planar + RPA & NaN & 51.2136 & 0.4618 & 0.5176 & 0.0480 & 0.3531 \\
Advancing-front + RPA & NaN & 57.5164 & 0.4913 & 0.5107 & 0.0503 & 0.3671 \\
Planar + LH-only & NaN & 4.1277 & 0.4721 & 0.4308 & 0.0468 & 0.3587 \\
\bottomrule
\end{tabular}



## Optional: combine Chair and Table pilot summaries

Run the notebook twice:
1. once with `chair_two_leg.yaml`
2. once with `table_two_leg.yaml`

Then execute the next cell to combine both summaries.


In [76]:
chair_csv = TABLES_DIR / "chair_two_leg_removal_multipart_pilot_summary.csv"
table_csv = TABLES_DIR / "table_two_leg_removal_multipart_pilot_summary.csv"

if chair_csv.exists() and table_csv.exists():
    chair_df = pd.read_csv(chair_csv)
    table_df = pd.read_csv(table_csv)

    chair_df.insert(0, "Setting", "Chair (two-leg)")
    table_df.insert(0, "Setting", "Table (two-leg)")

    combined = pd.concat([chair_df, table_df], ignore_index=True)
    combined_csv = TABLES_DIR / "multipart_pilot_combined.csv"
    combined_tex = TABLES_DIR / "multipart_pilot_combined.tex"

    combined.to_csv(combined_csv, index=False)
    combined_latex = combined.to_latex(index=False, float_format=lambda x: f"{x:.4f}")

    with open(combined_tex, "w", encoding="utf-8") as f:
        f.write(combined_latex)

    print("Saved combined CSV:", combined_csv)
    print("Saved combined TeX:", combined_tex)
    display(combined)
else:
    print("Run both chair and table pilot first.")
    print("Missing:", [str(p) for p in [chair_csv, table_csv] if not p.exists()])

Saved combined CSV: D:\MyJupyter\Works\3DPART_v2\outputs\tables\multipart_pilot_combined.csv
Saved combined TeX: D:\MyJupyter\Works\3DPART_v2\outputs\tables\multipart_pilot_combined.tex


,Setting,Method,Residual ↓,Improvement ↑,Quality ↑,Locality ↑,CD ↓,HD ↓
0,Chair (two-leg),Planar + RPA,0.9851,24.0703,0.4726,0.5530,0.0841,0.6102
1,Chair (two-leg),Advancing-front + RPA,0.0000,27.1032,0.4792,0.5184,0.0842,0.5885
2,Chair (two-leg),Planar + LH-only,22.3979,2.6575,0.4162,0.2248,0.0835,0.6121
3,Table (two-leg),Planar + RPA,NaN,51.2136,0.4618,0.5176,0.0480,0.3531
4,Table (two-leg),Advancing-front + RPA,NaN,57.5164,0.4913,0.5107,0.0503,0.3671
5,Table (two-leg),Planar + LH-only,NaN,4.1277,0.4721,0.4308,0.0468,0.3587


## Suggested paper wording

After running the chair and table pilots, you can add a short paragraph like:

> As a preliminary extension toward multi-part scenarios, we further evaluate two-leg removal on small chair and table subsets. Although the problem becomes more difficult than single-part removal, target-aware methods still outperform the largest-hole-only baseline in residual closure and locality. This pilot suggests that the proposed task formulation remains meaningful beyond single-part settings and motivates future construction of a larger multi-part benchmark.


In [77]:
RESULT_CSV = r"D:\MyJupyter\Works\3DPART_v2\outputs\table_two_leg_removal\metrics\all_results.csv"

import pandas as pd

raw_df = pd.read_csv(RESULT_CSV)

METHOD_SPECS = {
    "Planar + RPA": "planar_removed_part_aware",
    "Advancing-front + RPA": "advancing_front_rpa",
    "Planar + LH-only": "planar_largest_hole_only",
}

METRIC_MAP = {
    "Residual ↓": "closure_residual",
    "Improvement ↑": "improvement",
    "Quality ↑": "avg_new_face_quality",
    "Locality ↑": "locality_ratio",
    "CD ↓": "chamfer_distance",
    "HD ↓": "hausdorff_distance",
}

rows = []
for method_name, prefix in METHOD_SPECS.items():
    row = {"Method": method_name}
    for out_name, metric_name in METRIC_MAP.items():
        col = f"{prefix}/{metric_name}"
        if col in raw_df.columns:
            row[out_name] = pd.to_numeric(raw_df[col], errors="coerce").mean()
        else:
            row[out_name] = None
    rows.append(row)

pilot_table = pd.DataFrame(rows)

pilot_table_rounded = pilot_table.copy()
for c in pilot_table_rounded.columns:
    if c != "Method":
        pilot_table_rounded[c] = pd.to_numeric(
            pilot_table_rounded[c], errors="coerce"
        ).round(4)

pilot_table_rounded

,Method,Residual ↓,Improvement ↑,Quality ↑,Locality ↑,CD ↓,HD ↓
0,Planar + RPA,0.4548,51.2136,0.4618,0.5176,0.0480,0.3531
1,Advancing-front + RPA,0.0000,57.5164,0.4913,0.5107,0.0503,0.3671
2,Planar + LH-only,47.5408,4.1277,0.4721,0.4308,0.0468,0.3587


In [78]:
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

out_csv = TABLES_DIR / f"{EXP_NAME}_multipart_pilot_summary.csv"
out_tex = TABLES_DIR / f"{EXP_NAME}_multipart_pilot_summary.tex"

pilot_table_rounded.to_csv(out_csv, index=False)

latex_str = pilot_table_rounded.to_latex(index=False, float_format=lambda x: f"{x:.4f}")
with open(out_tex, "w", encoding="utf-8") as f:
    f.write(latex_str)

print("Saved CSV :", out_csv)
print("Saved TeX :", out_tex)
print(latex_str)

Saved CSV : D:\MyJupyter\Works\3DPART_v2\outputs\tables\table_two_leg_removal_multipart_pilot_summary.csv
Saved TeX : D:\MyJupyter\Works\3DPART_v2\outputs\tables\table_two_leg_removal_multipart_pilot_summary.tex
\begin{tabular}{lrrrrrr}
\toprule
Method & Residual ↓ & Improvement ↑ & Quality ↑ & Locality ↑ & CD ↓ & HD ↓ \\
\midrule
Planar + RPA & 0.4548 & 51.2136 & 0.4618 & 0.5176 & 0.0480 & 0.3531 \\
Advancing-front + RPA & 0.0000 & 57.5164 & 0.4913 & 0.5107 & 0.0503 & 0.3671 \\
Planar + LH-only & 47.5408 & 4.1277 & 0.4721 & 0.4308 & 0.0468 & 0.3587 \\
\bottomrule
\end{tabular}

